In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ── State ─────────────────────────────────────────────────────────────────────
class State(TypedDict):
    topic: str
    joke: str

In [11]:

# ── Node 1: Generate / Refine content ────────────────────────────────────────
def generate(state: State) -> dict:
    if state["joke"] == "":
        # First pass — write from scratch
        prompt = (
            f"Write a joke about the following topic.\n\n"
            f"Topic: {state['topic']}\n\n"
            f"Return only the joke."
        )

    response = llm.invoke([
        SystemMessage(content="You are a skilled writer who produces clear, engaging jokes."),
        HumanMessage(content=prompt),
    ])

    state["joke"] = response.content

    return state

In [12]:
# ── Build Graph ───────────────────────────────────────────────────────────────
builder = StateGraph(State)

builder.add_node("generate", generate)

# START → generate → END
builder.add_edge(START, "generate")
builder.add_edge("generate", END)

graph = builder.compile()
print("Graph compiled successfully.")


Graph compiled successfully.


In [16]:
# ── Prediction Step ──────────────────────────────────────────────────────────
input_state = {"topic": "ramesh", "joke": ""}
prediction = graph.invoke(input_state)

print("Prediction:")
print(prediction["joke"])

Prediction:
Why did Ramesh bring a ladder to the party? Because he heard the drinks were on the house! 🥤
